# tinygpt: pretraining on a free Colab T4

Trains the 17.3M-parameter model from [AbuSayeed917/tinygpt](https://github.com/AbuSayeed917/tinygpt)
on about 354M tokens of cleaned FineWeb-Edu.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU.**
2. Upload `train.bin`, `val.bin` and `meta.json` from `~/tinygpt-lm/data/tokens/` on your Mac to
   **`MyDrive/tinylm/tokens/`** in Google Drive.
3. **Runtime → Run all.**

**If Colab disconnects:** reconnect and use **Run all** again. Training resumes from the last
checkpoint in `MyDrive/tinylm/ckpt/pretrain-v1/` (saved every 10 minutes, and on Stop).

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime → Change runtime type → T4 GPU, then Run all again.")
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/tinylm"
TOKENS_ON_DRIVE = f"{DRIVE_DIR}/tokens"
CKPT_DIR = f"{DRIVE_DIR}/ckpt/pretrain-v1"
# Stop and save a little before Colab's session limit rather than being cut off.
MAX_MINUTES = 170

Get the code. The commit printed here is also stored in every checkpoint.

In [ ]:
!rm -rf /content/tinygpt && git clone --quiet --depth 1 https://github.com/AbuSayeed917/tinygpt /content/tinygpt
%cd /content/tinygpt
!git log -1 --format='commit %H  %s'
!pip install --quiet safetensors==0.8.0

Copy the token files from Drive to the VM's local disk: random reads from Drive are slow.

In [ ]:
import json
import os
import shutil

os.makedirs("/content/tokens", exist_ok=True)
for name in ("meta.json", "val.bin", "train.bin"):
    source, target = f"{TOKENS_ON_DRIVE}/{name}", f"/content/tokens/{name}"
    if not os.path.exists(source):
        raise FileNotFoundError(f"Upload data/tokens/{name} to {TOKENS_ON_DRIVE}/ first")
    if not os.path.exists(target) or os.path.getsize(target) != os.path.getsize(source):
        print(f"copying {name} ({os.path.getsize(source) / 1e9:.2f} GB)")
        shutil.copy(source, target)
print(json.load(open("/content/tokens/meta.json"))["tokens"])

## Train

Every 250 steps it prints validation loss, bits per byte, throughput and a short text
sample. The script checks the token files against `meta.json` and the tokenizer's sha256
before it starts.

In [ ]:
!python -m train.pretrain --data-dir /content/tokens --ckpt-dir "{CKPT_DIR}" --max-minutes {MAX_MINUTES}

## Loss curves

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

log = pd.read_csv(f"{CKPT_DIR}/log.csv")
figure, (loss_axis, bpb_axis) = plt.subplots(1, 2, figsize=(12, 4))
loss_axis.plot(log.tokens_seen / 1e6, log.train_loss, label="train")
loss_axis.plot(log.tokens_seen / 1e6, log.val_loss, label="validation")
loss_axis.set(xlabel="tokens seen (millions)", ylabel="cross-entropy loss", title="Loss")
loss_axis.legend()
bpb_axis.plot(log.tokens_seen / 1e6, log.val_bits_per_byte)
bpb_axis.set(xlabel="tokens seen (millions)", ylabel="bits per byte", title="Validation bits per byte")
plt.show()
log.tail()

## Try the best checkpoint

In [ ]:
from pathlib import Path

from bpe.tokenizer import ByteLevelBPE
from model.gpt import GPT, ModelConfig
from train.pretrain import load_checkpoint

state = load_checkpoint(Path(CKPT_DIR) / "best.pt")
model = GPT(ModelConfig(**state["model_config"])).cuda().eval()
model.load_state_dict(state["model"])
tokenizer = ByteLevelBPE.load(Path("bpe/tinygpt-bpe-8k.json"))
print(f"best.pt: step {state['step']:,}, val loss {state['best_val_loss']:.3f}")

torch.manual_seed(0)
for prompt in ("The water cycle", "In computer science, a function", "The history of London"):
    ids = torch.tensor([tokenizer.encode(prompt)], device="cuda")
    output = model.generate(ids, max_new_tokens=60, temperature=0.8, top_k=50)
    print(repr(tokenizer.decode(output[0].tolist())), end="\n\n")